In [41]:
from scipy.io import loadmat
import numpy as np
import torch

file_path = "data/BCICIV_calib_ds1a_1000Hz.mat"
device = torch.device('cuda') if torch.cuda.is_available() else torch.device("cpu")

mat = loadmat(file_path)

mat.keys()

dict_keys(['__header__', '__version__', '__globals__', 'cnt', 'mrk', 'nfo'])

In [42]:
freq = mat["nfo"]["fs"][0][0][0][0]
cls_names = mat["nfo"]["classes"][0][0][0]

image_display_time_s = 4
blank_display_time_s = 2
fixation_only_display_time_s = 2

eeg_signal = mat["cnt"]
n_channels = eeg_signal.shape[1]
n_samples =  eeg_signal.shape[0]
n_classes = 2
batch_size = 16

target_classes = mat["mrk"]["y"][0][0][0]
image_show_index = mat["mrk"]["pos"][0][0][0]

cls_names

array([array(['left'], dtype='<U4'), array(['foot'], dtype='<U4')],
      dtype=object)

Dataset 1 from BCI IV (BBCI) contains data for 7 subjects. For each subject two classes of motor imagery were selected from the three classes left hand, right hand, and foot.

## Data preparation
Cutting out from full EEG signal 4-second sequences (for the time when the cues were displayed)

Labels are initially -1/1. Changing to 0/1

In [43]:
trial_len = int(freq * image_display_time_s)  # 4 seconds

X_trials = []
y_trials = []
for pos, cls in zip(image_show_index, target_classes):
    start = pos
    end = pos + trial_len
    if end <= n_samples:
        X_trials.append(eeg_signal[start:end])  # (T, C)
        y_trials.append(int((cls+1)/2))

X_trials = torch.tensor(np.stack(X_trials)).float()  # (N_trials, T, C)
y_trials = torch.tensor(y_trials).long()

mean = X_trials.mean(dim=(0, 1), keepdim=True)
std = X_trials.std(dim=(0, 1), keepdim=True)
X_trials = (X_trials - mean) / (std + 1e-6)

In [44]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_trials, y_trials, test_size=0.2)

In [45]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [46]:
print("EEG signal:")
print(X_trials.shape)
print()
print("Classes corresponding to signal")
print(set(y_trials.numpy()))

EEG signal:
torch.Size([200, 4000, 59])

Classes corresponding to signal
{np.int64(0), np.int64(1)}


## Training

In [47]:
from torchesn.nn import ESN
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix

model = ESN(
    input_size=n_channels,
    hidden_size=64,
    num_layers=3,
    nonlinearity='tanh',
    output_size=n_classes,
    # output_steps="mean",
    output_steps="last",
    readout_training='cholesky',
    batch_first=True
).to(device)

transient_period = int(freq * 1) # 1 second
washout = torch.full((batch_size,), transient_period, device=device) # ignore first 1 second
for x, y in train_dataloader:
    x = x.to(device)
    y = y.to(device)
    model(x, washout, None, F.one_hot(y, n_classes).float())

model.fit()

### Train accuracy

In [48]:
train_preds = []
train_labels = []
for x, y in train_dataloader:
    with torch.no_grad():
        y_pred, _ = model(x, washout)
        pred_classes = y_pred[:, -1].argmax(dim=-1)
        train_labels.extend(y)
        train_preds.extend(pred_classes)

accuracy = (np.array(train_preds) == np.array(train_labels)).mean()
cm = confusion_matrix(train_labels, train_preds)
print(f"Accuracy: {accuracy.item():.4f}")
print(cm)

Accuracy: 0.8063
[[63 17]
 [14 66]]


### Test accuracy

In [49]:
test_dataset = TensorDataset(X_test, y_test)
test_dataloader = DataLoader(test_dataset, batch_size)

test_preds = []
test_labels = []
for x, y in test_dataloader:
    with torch.no_grad():
        y_pred, _ = model(x, washout)
        pred_classes = y_pred[:, -1].argmax(dim=-1)
        test_labels.extend(y)
        test_preds.extend(pred_classes)

accuracy = (np.array(test_preds) == np.array(test_labels)).mean()
cm = confusion_matrix(test_labels, test_preds)
print(f"Accuracy: {accuracy.item():.4f}")
print(cm)

Accuracy: 0.5500
[[ 9 11]
 [ 7 13]]


## Training with SGD (does not work)

In [ ]:
from sklearn.metrics import accuracy_score

model = ESN(
    input_size=n_channels,
    hidden_size=256,
    output_size=n_classes,
    output_steps="last",
    readout_training='gd',
    batch_first=True
).to(device)
optimizer = torch.optim.Adam(model.parameters())
for epoch in range(10):
    for i, batch in enumerate(train_dataloader):
        x, y = batch
        x = x.to(device)
        y = y.to(device)
        washout = torch.full((x.size(0),), transient_period, device=device) # ignore first 1s

        optimizer.zero_grad()
        output, _ = model(x, washout)
        loss = torch.nn.CrossEntropyLoss()(output, F.one_hot(y, n_classes))
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            y_pred, _ = model(x, washout)
            pred_classes = y_pred.argmax(dim=-1)
            accuracy = accuracy_score(pred_classes[:, -1], y)

        print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Accuracy: {accuracy:.4f}")